In [ ]:
import torch
import torchvision
from torchvision import datasets
from torchvision import transforms
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import DataLoader




In [ ]:
# here size of image isnt same for 0 -> 3,30,29 ; for 45-> 3,38,38
#so we need to generalise the image shape
train_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(10),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1)
    ),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor()
])
transform=transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor()
])

In [ ]:
train_dataset=datasets.GTSRB(
    root="data",
    split="train",
    download=True,
    transform=train_transform
)

test_dataset=datasets.GTSRB(
    root="data",
    split="test",
    download=True,
    transform=transform
)

len(train_dataset),len(test_dataset)

In [ ]:
image,label=train_dataset[0]

In [ ]:
print(image.shape)
print(type(image))
print(type(label))
print(label)

In [ ]:
print(dir(train_dataset))


In [ ]:
#matplotlib.pyplot.imshow expects images in the format (Height, Width, Channels) or (Height, Width) for grayscale, but the image tensor is in (Channels, Height, Width) format.
image,label=train_dataset[0]
plt.imshow(image.permute(1,2,0))
plt.title(f"Label: {label}")




In [ ]:
fig=plt.figure(figsize=(20,20))
rows,cols=5,5
for i in range(0,rows*cols):
    random_idx = torch.randint(0, len(train_dataset)-1,size=[1]).item()
    image,label=train_dataset[random_idx]
    fig.add_subplot(rows,cols,i+1)
    plt.imshow(image.permute(1,2,0))
    plt.title(f"Label: {label}")

In [ ]:
train_dataset,test_dataset

In [ ]:
#Data loader
train_dataloader=DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True
)
test_dataloader=DataLoader(
    dataset=test_dataset,
    batch_size=32,
    shuffle=True
)

In [ ]:
print(f"DataLoader : {train_dataloader}\n Length of train Dataloader: {len(train_dataloader)}")
print(f"DataLoader : {test_dataloader}\n Length of test Dataloader: {len(test_dataloader)}")

In [ ]:
labels = [label for _, label in train_dataset]
print(len(set(labels)))

In [ ]:
#Build Model

class GTSRBModel(nn.Module):
  def __init__(self,input_size:int,hidden_units:int,output_size:int):
     super().__init__()
     self.conv_block_1=nn.Sequential(
         nn.Conv2d(
             in_channels=input_size,
             out_channels=hidden_units,
             kernel_size=3,
             stride=1,
             padding=1
         ),
         nn.ReLU(),
         nn.MaxPool2d(kernel_size=2)
     )
     self.conv_block_2=nn.Sequential(
         nn.Conv2d(
             in_channels=hidden_units,
             out_channels=hidden_units*2,
             stride=1,
             kernel_size=3,
             padding=1
         ),
         nn.ReLU(),
         nn.MaxPool2d(kernel_size=2)
     )
     self.classifier=nn.Sequential(
         nn.Flatten(),
         nn.Linear(
             in_features=hidden_units*2*8*8,
             out_features=output_size
         )
     )
  def forward(self,x):
    x=self.conv_block_1(x)
    x=self.conv_block_2(x)
    x=self.classifier(x)
    return x



In [ ]:
model_0=GTSRBModel(
    input_size=3,
    hidden_units=32,
    output_size=43
)
model_0

In [ ]:
#loss and optimiser
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(
    params=model_0.parameters(),
    lr=0.001
)
print(f"Loss function: {loss_fn}")
print(f"Optimizer: {optimizer}")

In [ ]:
from tqdm.auto import tqdm

epochs=10
train_losses=[]
test_losses=[]
test_accs=[]
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_0.to(device)
for epoch in tqdm(range(epochs)):
  print(f"Epoch: {epoch}\n---")

  train_loss=0
  #Training
  model_0.train()
  #forward propagation in batches
  for X,y in train_dataloader:
    X,y=X.to(device),y.to(device)
    y_pred=model_0(X)
    #loss
    loss=loss_fn(y_pred,y)
    train_loss+=loss
    #optimizer zero grad
    optimizer.zero_grad()
    #backward propagation
    loss.backward()
    #optimizer tsep
    optimizer.step()
  train_loss/=len(train_dataloader)

  #Testing
  test_loss,test_acc=0,0
  model_0.eval()
  with torch.inference_mode():
    for X_test,y_test in test_dataloader:
      X_test,y_test=X_test.to(device),y_test.to(device)
      test_pred=model_0(X_test)
      test_loss += loss_fn(test_pred, y_test)
      test_acc += (test_pred.argmax(dim=1) == y_test).sum().item()
    test_loss /= len(test_dataloader)
    test_acc /= len(test_dataset)

  #save losses

  train_losses.append(train_loss.item())
  test_losses.append(test_loss.item())
  test_accs.append(test_acc)

  print(f"Train loss: {train_loss:.4f} | Test loss: {test_loss:.4f} | Test acc: {test_acc*100:.2f}%")





In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(len(train_losses))

plt.figure(figsize=(10, 5))

# Loss plot
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_losses, label='Train loss')
plt.plot(epochs_range, test_losses, label='Test loss')
plt.title('Loss curves')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Accuracy plot
plt.subplot(1, 2, 2)
plt.plot(epochs_range, test_accs, label='Test accuracy')
plt.title('Accuracy curve')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

#MODEL RESNET18

In [ ]:
from torchvision.models import resnet18,ResNet18_Weights

In [ ]:
train_transform_resnet = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

test_transform_resnet = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [ ]:
train_dataset_resnet = datasets.GTSRB(
    root="data",
    split="train",
    download=True,
    transform=train_transform_resnet
)

test_dataset_resnet = datasets.GTSRB(
    root="data",
    split="test",
    download=True,
    transform=test_transform_resnet
)

In [ ]:
train_dataloader_resnet = DataLoader(
    train_dataset_resnet,
    batch_size=32,
    shuffle=True
)

test_dataloader_resnet = DataLoader(
    test_dataset_resnet,
    batch_size=32,
    shuffle=False
)

In [ ]:
weights = ResNet18_Weights.DEFAULT

model_resnet = resnet18(weights=weights)

In [ ]:
print(model_resnet)

In [ ]:
print(model_resnet.fc)

In [ ]:
model_resnet.fc = nn.Linear(
    in_features=model_resnet.fc.in_features,
    out_features=43
)

In [ ]:
model_resnet.to(device)

In [ ]:
loss_fn_resnet = nn.CrossEntropyLoss()
optimizer_resnet = torch.optim.Adam(model_resnet.parameters(),lr=0.0001)

In [ ]:
from tqdm.auto import tqdm

epochs=3
train_losses_resnet=[]
test_losses_resnet=[]
test_accs_resnet=[]
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_resnet.to(device)
for epoch in tqdm(range(epochs)):
  print(f"Epoch: {epoch}\n---")

  train_loss=0
  #Training
  model_resnet.train()
  #forward propagation in batches
  for X,y in train_dataloader_resnet:
    X,y=X.to(device),y.to(device)
    y_pred=model_resnet(X)
    #loss
    loss=loss_fn_resnet(y_pred,y)
    train_loss+=loss.item()
    #optimizer zero grad
    optimizer_resnet.zero_grad()
    #backward propagation
    loss.backward()
    #optimizer tsep
    optimizer_resnet.step()
  train_loss/=len(train_dataloader_resnet)

  #Testing
  test_loss,test_acc=0,0
  model_resnet.eval()
  with torch.inference_mode():
    for X_test,y_test in test_dataloader_resnet:
      X_test,y_test=X_test.to(device),y_test.to(device)
      test_pred=model_resnet(X_test)
      test_loss += loss_fn_resnet(test_pred, y_test).item()
      test_acc += (test_pred.argmax(dim=1) == y_test).sum().item()
    test_loss /= len(test_dataloader_resnet)
    test_acc /= len(test_dataset_resnet)

  #save losses

  train_losses_resnet.append(train_loss)
  test_losses_resnet.append(test_loss)
  test_accs_resnet.append(test_acc)

  print(f"Train loss: {train_loss:.4f} | Test loss: {test_loss:.4f} | Test acc: {test_acc*100:.2f}%")

In [ ]:
torch.save(model_resnet.state_dict(), "resnet18_gtsrb.pth")

print("Model Saved Successfully!")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

model_resnet.eval()

y_true = []
y_pred = []

with torch.inference_mode():
    for X, y in test_dataloader_resnet:
        X = X.to(device)
        y = y.to(device)

        outputs = model_resnet(X)

        preds = outputs.argmax(dim=1)

        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

In [ ]:
print(classification_report(y_true, y_pred))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(40,40))

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues", xticks_rotation=90)

plt.title("Confusion Matrix")
plt.show()

In [ ]:
import random

fig, axes = plt.subplots(3,3, figsize=(12,12))

model_resnet.eval()

for ax in axes.flat:

    idx = random.randint(0, len(test_dataset_resnet)-1)

    image, label = test_dataset_resnet[idx]

    with torch.inference_mode():

        pred = model_resnet(
            image.unsqueeze(0).to(device)
        ).argmax(dim=1).item()

    ax.imshow(image.permute(1,2,0))
    ax.set_title(f"Pred: {pred}\nTrue: {label}")

    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(train_losses_resnet,label="Train Loss")
plt.plot(test_losses_resnet,label="Test Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.savefig("loss_curve_resnet.png")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot([x*100 for x in test_accs_resnet])

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")

plt.savefig("accuracy_curve_resnet.png")

plt.show()

In [ ]:
from google.colab import files
files.download("resnet18_gtsrb.pth")

In [ ]:
from google.colab import files

#
files.download("accuracy_curve_resnet.png")

In [ ]:
%%writefile requirements.txt
torch
torchvision
numpy
matplotlib
scikit-learn
tqdm

In [ ]:
from google.colab import files
files.download("requirements.txt")